# RT-DETR

[Lv et al. (2024), DETRs Beat YOLOs on Real-time Object Detection](https://arxiv.org/abs/2304.08069)

DETR・Deformable DETRはNMSフリーというアーキテクチャ上の利点を持ちながらも、実際の推論速度ではYOLO系のリアルタイム検出器に及ばなかった。一方YOLO系は高速だが、NMSが推論のボトルネックになるうえ、NMSのIoU閾値などのハイパーパラメータがモデル自体の速度・精度トレードオフに影響してしまう問題があった。

**RT-DETR（Real-Time DEtection TRansformer）** は、DETR系のNMSフリーという利点を保ちながら、YOLO系に匹敵する推論速度を実現した最初のTransformerベース検出器。主な工夫は **Efficient Hybrid Encoder** と **IoU-aware Query Selection** の2つ。

## Efficient Hybrid Encoder

Deformable DETRは計算量を抑えるためにDeformable Attentionを導入したが、それでも複数スケールの特徴マップすべてに対してTransformer Encoderを適用しており、これが推論速度上のボトルネックの一つになっていた。

RT-DETRは、特徴マップ間の相互作用を次の2種類に分解し、それぞれに適した処理を割り当てる。

1. **Intra-scale Feature Interaction（同一スケール内の相互作用）**：**最も解像度の低い（意味的に最も豊かな）特徴マップ1つだけ** に、通常のTransformer Encoder（Self-Attention、AIFIと呼ばれる）を適用する。解像度が低いため計算コストが小さく、かつ物体検出で重要な大域的な文脈・物体間の関係はこのレベルで十分に捉えられるという観察に基づく
2. **Cross-scale Feature Fusion（スケール間の融合）**：複数解像度の特徴マップ間の統合は、Attentionではなく**CNNベースの軽量なモジュール（CCFM）**で行う。PANetのようなtop-down／bottom-upの畳み込みパスで特徴を融合する

つまり「重いAttentionは最小限の（解像度が最も低い）特徴マップだけに限定し、残りはCNNの得意な軽量な畳み込みで済ませる」という役割分担により、Deformable DETRの多スケールDeformable Attentionと同等以上の表現力を、大幅に少ない計算量で実現している。

## IoU-aware Query Selection

Deformable DETRのTwo-stage方式では、Encoderの出力特徴を分類スコアでランキングし、上位$N$個をDecoderのobject query（の初期値）として選択していた。しかしこの分類スコアは「クラスらしさ」を反映するだけで、「ボックスの位置精度（IoU）」を直接反映していないため、**分類スコアは高いがボックス位置は不正確**な特徴量が選ばれてしまうことがあった（分類と局在化の不整合）。

RT-DETRは、Encoderの学習時に分類損失だけでなくIoUを考慮した目的関数を使うことで、クラス予測とボックス品質（IoU）の両方が高い特徴量ほど高いスコアを持つように学習する（**IoU-aware Query Selection**）。これにより、Decoderに渡される初期queryの質が向上し、Decoderの層数が少なくても高い精度を達成できるようになる。

## 速度と精度のオンデマンドな調整

RT-DETRのDecoderは複数層のTransformer層を積み重ねた構造だが、実装上、**推論時に使うDecoder層の数を後から変更できる**ように設計されている（再学習不要）。層数を減らせば速度が上がり精度がやや落ち、増やせば逆になるため、1つの学習済みモデルから、デプロイ先のハードウェアや要求速度に応じて速度・精度のトレードオフを柔軟に選べる。

## まとめ：DETR系譜における位置づけ

| | DETR | Deformable DETR | RT-DETR |
| --- | --- | --- | --- |
| Attention対象 | 全画素（単一スケール） | 学習されたK点（マルチスケール） | 最低解像度のみTransformer、残りはCNNで融合 |
| Query選択 | ランダム初期化 | 分類スコアに基づくTwo-stage選択 | IoUを考慮したQuery選択 |
| 主な課題への対応 | — | 収束の遅さ・小物体検出 | 推論速度（YOLO並みのリアルタイム性） |
| NMS | 不要 | 不要 | 不要 |

DETR→Deformable DETR→RT-DETRという流れは、「集合予測によりNMSを排除する」という利点を保ちながら、「収束速度」「小物体への対応」「推論速度」という実用上の課題を1つずつ解決していく進化として整理できる。

## 参考文献

- Lv, W. et al. (2024). [DETRs Beat YOLOs on Real-time Object Detection (RT-DETR)](https://arxiv.org/abs/2304.08069)
- [lyuwenyu/RT-DETR — GitHub](https://github.com/lyuwenyu/RT-DETR)
- Ultralyticsによる実装：[RT-DETR - Ultralytics YOLO Docs](https://docs.ultralytics.com/models/rtdetr/)（`pip install ultralytics` で `from ultralytics import RTDETR` として利用可能）